# Explainable AI for Physics-Informed PMSM Temperature Prediction

Objective:
- Explain model predictions
- Compare different models
- Validate explanations using physics

In [1]:
# ============================================================
# Standard Libraries
# ============================================================

import os
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# Numerical Libraries
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# Visualization
# ============================================================

import matplotlib.pyplot as plt
import matplotlib

plt.style.use("default")
matplotlib.rcParams["figure.figsize"] = (10,6)
matplotlib.rcParams["font.size"] = 12

# ============================================================
# Machine Learning
# ============================================================

import joblib

from sklearn.preprocessing import StandardScaler

# ============================================================
# Explainable AI
# ============================================================

import shap

# ============================================================
# Deep Learning
# ============================================================

import torch
import torch.nn as nn

# ============================================================
# Notebook Display
# ============================================================

from IPython.display import display

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random Seed:", SEED)

Random Seed: 42


In [3]:
# ============================================================
# Device Configuration
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Running on:", DEVICE)

if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

Running on: cuda
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [4]:
# ============================================================
# Project Directory
# ============================================================

PROJECT_DIR = Path(r"D:\Work space\PIML_Motor")

DATA_DIR = PROJECT_DIR / "data"

MODEL_DIR = PROJECT_DIR / "models"

OUTPUT_DIR = PROJECT_DIR / "Explainability"

OUTPUT_DIR.mkdir(exist_ok=True)

print(PROJECT_DIR)

D:\Work space\PIML_Motor


In [5]:
# ============================================================
# Output Directories
# ============================================================

FIGURE_DIR = OUTPUT_DIR / "Figures"

CSV_DIR = OUTPUT_DIR / "CSV"

REPORT_DIR = OUTPUT_DIR / "Reports"

for folder in [FIGURE_DIR, CSV_DIR, REPORT_DIR]:
    folder.mkdir(exist_ok=True)

print("Folders Ready")

Folders Ready


In [6]:
# ============================================================
# Load Dataset
# ============================================================

DATASET_PATH = DATA_DIR / "D:\\Work space\\PIML_Motor\\dataset\\measures_v2.csv"

df = pd.read_csv(DATASET_PATH)

print(df.shape)

df.head()

(1330816, 13)


,u_q,coolant,stator_winding,u_d,stator_tooth,motor_speed,i_d,i_q,pm,stator_yoke,ambient,torque,profile_id
0,-0.450682,18.805172,19.086670,-0.350055,18.293219,0.002866,0.004419,0.000328,24.554214,18.316547,19.850691,0.187101,17
1,-0.325737,18.818571,19.092390,-0.305803,18.294807,0.000257,0.000606,-0.000785,24.538078,18.314955,19.850672,0.245417,17
2,-0.440864,18.828770,19.089380,-0.372503,18.294094,0.002355,0.001290,0.000386,24.544693,18.326307,19.850657,0.176615,17
3,-0.327026,18.835567,19.083031,-0.316199,18.292542,0.006105,0.000026,0.002046,24.554018,18.330833,19.850647,0.238303,17
4,-0.471150,18.857033,19.082525,-0.332272,18.291428,0.003133,-0.064317,0.037184,24.565397,18.326662,19.850639,0.208197,17


In [7]:
print("="*50)

print(df.info())

print("="*50)

display(df.describe().T)

<class 'pandas.DataFrame'>
RangeIndex: 1330816 entries, 0 to 1330815
Data columns (total 13 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   u_q             1330816 non-null  float64
 1   coolant         1330816 non-null  float64
 2   stator_winding  1330816 non-null  float64
 3   u_d             1330816 non-null  float64
 4   stator_tooth    1330816 non-null  float64
 5   motor_speed     1330816 non-null  float64
 6   i_d             1330816 non-null  float64
 7   i_q             1330816 non-null  float64
 8   pm              1330816 non-null  float64
 9   stator_yoke     1330816 non-null  float64
 10  ambient         1330816 non-null  float64
 11  torque          1330816 non-null  float64
 12  profile_id      1330816 non-null  int64  
dtypes: float64(12), int64(1)
memory usage: 132.0 MB
None


,count,mean,std,min,25%,50%,75%,max
u_q,1330816.0,54.279005,44.173234,-25.290930,12.069925,48.938181,90.034393,133.036994
coolant,1330816.0,36.229995,21.786149,10.623751,18.698135,26.900141,49.857487,101.598512
stator_winding,1330816.0,66.342745,28.672061,18.585815,42.787957,65.110129,88.141137,141.362885
u_d,1330816.0,-25.133809,63.091972,-131.530411,-78.690904,-7.429755,1.470271,131.469788
stator_tooth,1330816.0,56.878582,22.952233,18.133976,38.416012,56.036345,75.586682,111.946423
motor_speed,1330816.0,2202.080728,1859.663350,-275.549144,317.110665,1999.976685,3760.639058,6000.015137
i_d,1330816.0,-68.716810,64.933233,-278.003632,-115.406060,-51.093763,-2.979688,0.051897
i_q,1330816.0,37.412782,92.181880,-293.426793,1.095863,15.774007,100.612150,301.707855
pm,1330816.0,58.506776,19.001498,20.856956,43.151582,60.266287,72.008371,113.606628
stator_yoke,1330816.0,48.187965,19.991004,18.076689,31.990333,45.625509,61.460828,101.147964


In [8]:
missing = df.isnull().sum()

display(
    missing[missing>0]
)

print("Total Missing Values:", missing.sum())

Series([], dtype: int64)

Total Missing Values: 0


In [9]:
TARGET = "stator_winding"

print("Target Variable:", TARGET)

Target Variable: stator_winding


In [10]:
FEATURES = [

'u_q',

'coolant',

'u_d',

'motor_speed',

'i_d',

'i_q',

'pm',

'stator_yoke',

'ambient',

'torque',

'stator_tooth'

]

print("Total Features:", len(FEATURES))

FEATURES

Total Features: 11


['u_q',
 'coolant',
 'u_d',
 'motor_speed',
 'i_d',
 'i_q',
 'pm',
 'stator_yoke',
 'ambient',
 'torque',
 'stator_tooth']

In [11]:
X = df[FEATURES]

y = df[TARGET]

print(X.shape)

print(y.shape)

(1330816, 11)
(1330816,)


In [13]:
X_SCALER_PATH = MODEL_DIR / "X_scaler.pkl"

x_scaler = joblib.load(X_SCALER_PATH)

print("X Scaler Loaded Successfully")

X Scaler Loaded Successfully


In [14]:
Y_SCALER_PATH = MODEL_DIR / "y_scaler.pkl"

y_scaler = joblib.load(Y_SCALER_PATH)

print("Y Scaler Loaded Successfully")

Y Scaler Loaded Successfully


In [15]:
FEATURES = list(df.columns)

FEATURES.remove(TARGET)

if "profile_id" in FEATURES:
    FEATURES.remove("profile_id")

print(FEATURES)

['u_q', 'coolant', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque']


In [16]:
X = df[FEATURES]

y = df[TARGET]

In [17]:
X_SCALER_PATH = MODEL_DIR / "X_scaler.pkl"

Y_SCALER_PATH = MODEL_DIR / "y_scaler.pkl"

x_scaler = joblib.load(X_SCALER_PATH)

y_scaler = joblib.load(Y_SCALER_PATH)

print("Scalers Loaded")

Scalers Loaded


In [19]:
X_scaled = x_scaler.transform(X)

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Copper_Loss
- Current_Magnitude
- Mechanical_Power
- Temp_Difference
- Thermal_Load_Index
- ...
